# 8.0 Fusion MVP Demo

Purpose:
- Demonstrate interpretable late fusion.
- Combine face, speech, acute symptoms, and metadata context.
- Show how final screening score and risk band are produced.

This notebook complements the reusable fusion scenario audit and reflects the audited deterministic late-fusion logic.

In [1]:
from pathlib import Path
import pandas as pd
import subprocess
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FUSION_MVP_DEMO_PATH,
    FUSION_MVP_REPORT_PATH,
)

from rural_stroke_assist.modules.fusion_module import (
    FusionInput,
    fuse_multimodal_scores,
    DEFAULT_FUSION_WEIGHTS,
)

In [2]:
demo_cases = [
    {
        "case_name": "low_concern",
        "face_acute_score": 0.10,
        "speech_acute_score": 0.15,
        "acute_symptom_score": 0.00,
        "metadata_contextual_risk_score": 0.20,
        "acute_symptom_hard_escalation": False,
    },
    {
        "case_name": "high_metadata_only",
        "face_acute_score": 0.12,
        "speech_acute_score": 0.18,
        "acute_symptom_score": 0.00,
        "metadata_contextual_risk_score": 0.90,
        "acute_symptom_hard_escalation": False,
    },
    {
        "case_name": "speech_dominant_case",
        "face_acute_score": 0.25,
        "speech_acute_score": 0.92,
        "acute_symptom_score": 0.35,
        "metadata_contextual_risk_score": 0.40,
        "acute_symptom_hard_escalation": False,
    },
    {
        "case_name": "face_and_speech_high",
        "face_acute_score": 0.88,
        "speech_acute_score": 0.81,
        "acute_symptom_score": 0.35,
        "metadata_contextual_risk_score": 0.55,
        "acute_symptom_hard_escalation": False,
    },
    {
        "case_name": "urgent_fast_case",
        "face_acute_score": 0.40,
        "speech_acute_score": 0.45,
        "acute_symptom_score": 0.85,
        "metadata_contextual_risk_score": 0.30,
        "acute_symptom_hard_escalation": True,
    },
    {
        "case_name": "missing_face_case",
        "face_acute_score": None,
        "speech_acute_score": 0.80,
        "acute_symptom_score": 0.35,
        "metadata_contextual_risk_score": 0.60,
        "acute_symptom_hard_escalation": False,
    },
]

In [3]:
rows = []

for case in demo_cases:
    case_name = case["case_name"]
    payload = {k: v for k, v in case.items() if k != "case_name"}

    result = fuse_multimodal_scores(
        FusionInput(**payload),
        weights=DEFAULT_FUSION_WEIGHTS,
    )

    rows.append(
        {
            "case_name": case_name,
            **payload,
            "fused_score": result.fused_score,
            "risk_band": result.risk_band,
            "modality_contributions": result.modality_contributions,
            "normalized_weights_used": result.normalized_weights_used,
            "warnings": " | ".join(result.warnings),
        }
    )

fusion_demo_df = pd.DataFrame(rows)

fusion_demo_df

,case_name,face_acute_score,speech_acute_score,acute_symptom_score,metadata_contextual_risk_score,acute_symptom_hard_escalation,fused_score,risk_band,modality_contributions,normalized_weights_used,warnings
0,low_concern,0.10,0.15,0.00,0.20,False,0.100000,LOW,"{'face': 0.035, 'speech': 0.045000000000000005...","{'face': 0.35000000000000003, 'speech': 0.3000...",Fusion output is a research screening estimate...
1,high_metadata_only,0.12,0.18,0.00,0.90,False,0.186000,LOW,"{'face': 0.042, 'speech': 0.054000000000000006...","{'face': 0.35000000000000003, 'speech': 0.3000...",Fusion output is a research screening estimate...
2,speech_dominant_case,0.25,0.92,0.35,0.40,False,0.500000,MODERATE,"{'face': 0.08750000000000001, 'speech': 0.2760...","{'face': 0.35000000000000003, 'speech': 0.3000...",Fusion output is a research screening estimate...
3,face_and_speech_high,0.88,0.81,0.35,0.55,False,0.750000,HIGH,"{'face': 0.30800000000000005, 'speech': 0.2430...","{'face': 0.35000000000000003, 'speech': 0.3000...",Fusion output is a research screening estimate...
4,urgent_fast_case,0.40,0.45,0.85,0.30,True,0.850000,URGENT,"{'face': 0.14, 'speech': 0.13500000000000004, ...","{'face': 0.35000000000000003, 'speech': 0.3000...",Fusion output is a research screening estimate...
5,missing_face_case,NaN,0.80,0.35,0.60,False,0.596154,MODERATE,"{'speech': 0.36923076923076925, 'acute_symptom...","{'speech': 0.4615384615384615, 'acute_symptoms...",Fusion output is a research screening estimate...


In [4]:
FUSION_MVP_DEMO_PATH.parent.mkdir(parents=True, exist_ok=True)

fusion_demo_df.to_csv(FUSION_MVP_DEMO_PATH, index=False)

FUSION_MVP_DEMO_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/fusion_mvp_demo_cases.csv')

In [5]:
audit_script = Path.cwd().parent / "scripts" / "fusion_scenario_audit.py"

subprocess.run([sys.executable, str(audit_script)], check=True)

audit_df = pd.read_csv(Path.cwd().parent / "reports" / "experiments" / "fusion_scenario_audit.csv")

audit_df[
    (audit_df["config_name"] == "A_current")
    & (audit_df["threshold_name"] == "M50_H75")
][[
    "scenario_name",
    "fused_score",
    "risk_band",
    "expected_min_band",
    "passes_expectation",
]].head(12)

,scenario_name,fused_score,risk_band,expected_min_band,passes_expectation
23,all_scores_low,0.110000,LOW,LOW,True
24,metadata_high_only_acute_low,0.181000,LOW,LOW,True
25,missing_one_modality_remaining_low,0.132308,LOW,LOW,True
26,face_high_only,0.383000,LOW,LOW,True
27,speech_high_only,0.356000,LOW,LOW,True
28,acute_symptoms_high_only,0.311000,LOW,LOW,True
29,metadata_high_only,0.140000,LOW,LOW,True
30,two_acute_modalities_moderate,0.500000,MODERATE,MODERATE,True
31,face_and_speech_high,0.750000,HIGH,HIGH,True
32,face_high_plus_symptoms_high,0.750000,HIGH,HIGH,True
